# 08 — LOGO evaluation for baseline CNN and frequency models

Evaluate our three models when one AI generator is completely unseen during training and validation. This notebook deliberately reuses the exact LOGO manifest built for ResNet18 and ViT, making all five model families directly comparable.

Run the notebook once per generator by changing `UNSEEN_GENERATORS` in the setup cell. Do not run multiple folds in parallel because each CNN training needs its own GPU/CPU resources.


## 1. Setup

The standard project hyperparameters remain fixed. Only the held-out generator changes between runs. Checkpoints and fitted frequency models are written to ignored folders; metrics, predictions, and figures are retained as experiment evidence.


In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.optim import AdamW


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "experiments.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT))

from src.data.config import load_config
from src.data.loaders import create_dataloaders
from src.data.logo import build_logo_manifest, list_ai_generators
from src.evaluation import (
    evaluate_predictions,
    evaluate_probability_predictions,
    save_evaluation_outputs,
)
from src.features import (
    compute_log_power_grid,
    compute_normalized_power_spectrum,
    compute_radial_energy_profile,
    image_to_luminance,
)
from src.models import BaselineCNN, count_trainable_parameters
from src.training import fit, set_seed

config = load_config()
SEED = config["seed"]
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Change only this list before starting the next LOGO fold.
UNSEEN_GENERATORS = ["wukong"]
UNSEEN_LABEL = "+".join(UNSEEN_GENERATORS)

MANIFEST_PATH = REPO_ROOT / "data" / "dataset.csv"
PROCESSED_ROOT = REPO_ROOT / "data" / "processed"
CHECKPOINT_ROOT = REPO_ROOT / "results" / "checkpoints" / f"logo_{UNSEEN_LABEL}"
MODEL_ROOT = REPO_ROOT / "results" / "models" / f"logo_{UNSEEN_LABEL}"
METRICS_ROOT = REPO_ROOT / "results" / "metrics" / f"logo_{UNSEEN_LABEL}"
FIGURES_ROOT = REPO_ROOT / "results" / "figures" / f"logo_{UNSEEN_LABEL}"

for path in (CHECKPOINT_ROOT, MODEL_ROOT, METRICS_ROOT, FIGURES_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print(f"Unseen generator(s): {UNSEEN_GENERATORS}")


Device: cpu
Unseen generator(s): ['wukong']


## 2. Build and verify the shared LOGO split

All AI images made by the held-out generator are moved to test. AI test images from seen generators are excluded. Real images retain the same splits used by the colleague's ResNet18/ViT experiment.

The assertions are essential: they prove that the held-out AI generator cannot influence training, validation, early stopping, or Logistic Regression model selection.


In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
all_generators = list_ai_generators(manifest)
assert all(g in all_generators for g in UNSEEN_GENERATORS)

logo_manifest = build_logo_manifest(manifest, UNSEEN_GENERATORS)

train_ai_generators = sorted(logo_manifest.loc[
    (logo_manifest["split"] == "train") & (logo_manifest["label"] == "ai"), "generator"
].unique())
val_ai_generators = sorted(logo_manifest.loc[
    (logo_manifest["split"] == "val") & (logo_manifest["label"] == "ai"), "generator"
].unique())
test_ai_generators = sorted(logo_manifest.loc[
    (logo_manifest["split"] == "test") & (logo_manifest["label"] == "ai"), "generator"
].unique())

assert not (set(UNSEEN_GENERATORS) & set(train_ai_generators))
assert not (set(UNSEEN_GENERATORS) & set(val_ai_generators))
assert set(test_ai_generators) == set(UNSEEN_GENERATORS)

display(logo_manifest.groupby(["split", "label"]).size().rename("images"))
print("Train AI generators:", train_ai_generators)
print("Val AI generators:  ", val_ai_generators)
print("Test AI generators: ", test_ai_generators)
print("LOGO leakage checks passed.")


split  label
test   ai       1000
       real     1050
train  ai       4200
       real     4900
val    ai        900
       real     1050
Name: images, dtype: int64

Train AI generators: ['adm', 'biggan', 'glide', 'midjourney', 'sdv1_5', 'vqdm']
Val AI generators:   ['adm', 'biggan', 'glide', 'midjourney', 'sdv1_5', 'vqdm']
Test AI generators:  ['wukong']
LOGO leakage checks passed.


## 3. Baseline CNN

This is the same four-block CNN, normalization, optimizer, and early-stopping configuration used in notebook 03. It is trained from scratch for this fold. Validation loss chooses the best epoch; test remains untouched until that choice is complete.


In [3]:
loader_config = config["dataloader"]
cnn_loaders = create_dataloaders(
    manifest=logo_manifest,
    processed_root=PROCESSED_ROOT,
    batch_size=loader_config["batch_size"],
    num_workers=loader_config["num_workers"],
    seed=SEED,
    pin_memory=device.type == "cuda",
)

model_config = config["baseline_cnn"]
cnn_model = BaselineCNN(
    channels=tuple(model_config["channels"]),
    dropout=model_config["dropout"],
).to(device)
cnn_criterion = nn.BCEWithLogitsLoss()
cnn_optimizer = AdamW(
    cnn_model.parameters(),
    lr=model_config["learning_rate"],
    weight_decay=model_config["weight_decay"],
)

print("Dataset sizes:", {split: len(loader.dataset) for split, loader in cnn_loaders.items()})
print(f"Trainable parameters: {count_trainable_parameters(cnn_model):,}")


Dataset sizes: {'train': 9100, 'val': 1950, 'test': 2050}
Trainable parameters: 389,153


In [4]:
# This is the long-running CNN training cell.
cnn_fit_result = fit(
    model=cnn_model,
    train_dataloader=cnn_loaders["train"],
    val_dataloader=cnn_loaders["val"],
    criterion=cnn_criterion,
    optimizer=cnn_optimizer,
    device=device,
    epochs=model_config["epochs"],
    patience=model_config["early_stopping_patience"],
    min_delta=model_config["early_stopping_min_delta"],
    checkpoint_path=CHECKPOINT_ROOT / "baseline_cnn.pt",
)
print(
    f"Best epoch: {cnn_fit_result.best_epoch}; "
    f"best validation loss: {cnn_fit_result.best_val_loss:.4f}"
)


Epoch 01/20 | train loss 0.6409, F1 0.6217 | val loss 0.5487, F1 0.7027
Epoch 02/20 | train loss 0.5613, F1 0.6924 | val loss 0.5350, F1 0.7190
Epoch 03/20 | train loss 0.5421, F1 0.6982 | val loss 0.6451, F1 0.6911
Epoch 04/20 | train loss 0.5323, F1 0.7077 | val loss 0.5390, F1 0.6944
Epoch 05/20 | train loss 0.5130, F1 0.7208 | val loss 0.5233, F1 0.7312
Epoch 06/20 | train loss 0.5122, F1 0.7210 | val loss 0.5047, F1 0.7403
Epoch 07/20 | train loss 0.4944, F1 0.7361 | val loss 0.4935, F1 0.7125
Epoch 08/20 | train loss 0.4935, F1 0.7392 | val loss 0.4949, F1 0.7059
Epoch 09/20 | train loss 0.4827, F1 0.7431 | val loss 0.4760, F1 0.7311
Epoch 10/20 | train loss 0.4771, F1 0.7514 | val loss 0.4797, F1 0.7606
Epoch 11/20 | train loss 0.4757, F1 0.7491 | val loss 0.4665, F1 0.7311
Epoch 12/20 | train loss 0.4653, F1 0.7523 | val loss 0.4651, F1 0.7445
Epoch 13/20 | train loss 0.4654, F1 0.7597 | val loss 0.4602, F1 0.7506
Epoch 14/20 | train loss 0.4593, F1 0.7623 | val loss 0.4626, F1

In [5]:
cnn_test_result = evaluate_predictions(
    model=cnn_model,
    dataloader=cnn_loaders["test"],
    criterion=cnn_criterion,
    device=device,
    threshold=0.5,
)
display(pd.Series(cnn_test_result.metrics.to_dict(), name="baseline_cnn"))

cnn_saved_paths = save_evaluation_outputs(
    cnn_test_result,
    output_dir=METRICS_ROOT / "baseline_cnn",
    figures_dir=FIGURES_ROOT / "baseline_cnn",
    model_label="Baseline CNN",
)
print("CNN outputs saved.")


loss         0.742601
accuracy     0.627805
precision    0.715847
recall       0.393000
f1           0.507424
roc_auc      0.717647
Name: baseline_cnn, dtype: float64

CNN outputs saved.


## 4. Frequency representations

Each image is transformed to luminance, then to a normalized FFT power spectrum. The radial model uses 64 annular energy bands. The hybrid model combines those 64 values with a 16×16 log-power grid, for 320 features.

Feature extraction is deterministic and contains no learning. The scaler and Logistic Regression are fitted only on train. Candidate regularization strengths are compared only on validation loss.


In [6]:
RADIAL_BINS = 64
GRID_SIZE = 16
THRESHOLD = 0.5


def extract_frequency_features(rows: pd.DataFrame):
    radial = np.empty((len(rows), RADIAL_BINS), dtype=np.float32)
    grid = np.empty((len(rows), GRID_SIZE * GRID_SIZE), dtype=np.float32)

    for index, filename in enumerate(rows["filename"]):
        with Image.open(PROCESSED_ROOT / filename) as image:
            luminance = image_to_luminance(image)
        power = compute_normalized_power_spectrum(luminance)
        radial[index] = compute_radial_energy_profile(power, bins=RADIAL_BINS)
        grid[index] = compute_log_power_grid(power, grid_size=GRID_SIZE).ravel()

        if (index + 1) % 1000 == 0 or index + 1 == len(rows):
            print(f"  extracted {index + 1:,}/{len(rows):,}")

    hybrid = np.concatenate((radial, grid), axis=1)
    labels = rows["label"].map({"real": 0, "ai": 1}).to_numpy(dtype=np.int64)
    return radial, hybrid, labels


def build_frequency_model(c_value: float) -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=c_value, max_iter=3000, random_state=SEED)),
    ])


def select_regularization(X_train, y_train, X_val, y_val, candidates):
    rows = []
    fitted = {}
    for c_value in candidates:
        candidate = build_frequency_model(c_value)
        candidate.fit(X_train, y_train)
        probabilities = candidate.predict_proba(X_val)[:, 1]
        result = evaluate_probability_predictions(
            logo_manifest[logo_manifest["split"] == "val"].reset_index(drop=True),
            probabilities,
            threshold=THRESHOLD,
        )
        rows.append({"C": c_value, **result.metrics.to_dict()})
        fitted[c_value] = candidate

    table = pd.DataFrame(rows).sort_values("loss").reset_index(drop=True)
    best_c = float(table.iloc[0]["C"])
    return fitted[best_c], best_c, table


In [7]:
# This is the long-running FFT extraction cell.
train_frame = logo_manifest[logo_manifest["split"] == "train"].reset_index(drop=True)
val_frame = logo_manifest[logo_manifest["split"] == "val"].reset_index(drop=True)
test_frame = logo_manifest[logo_manifest["split"] == "test"].reset_index(drop=True)

print("Train:")
X_train_radial, X_train_hybrid, y_train = extract_frequency_features(train_frame)
print("Validation:")
X_val_radial, X_val_hybrid, y_val = extract_frequency_features(val_frame)
print("Test:")
X_test_radial, X_test_hybrid, y_test = extract_frequency_features(test_frame)

print("Radial:", X_train_radial.shape, X_val_radial.shape, X_test_radial.shape)
print("Hybrid:", X_train_hybrid.shape, X_val_hybrid.shape, X_test_hybrid.shape)


Train:
  extracted 1,000/9,100
  extracted 2,000/9,100
  extracted 3,000/9,100
  extracted 4,000/9,100
  extracted 5,000/9,100
  extracted 6,000/9,100
  extracted 7,000/9,100
  extracted 8,000/9,100
  extracted 9,000/9,100
  extracted 9,100/9,100
Validation:
  extracted 1,000/1,950
  extracted 1,950/1,950
Test:
  extracted 1,000/2,050
  extracted 2,000/2,050
  extracted 2,050/2,050
Radial: (9100, 64) (1950, 64) (2050, 64)
Hybrid: (9100, 320) (1950, 320) (2050, 320)


In [8]:
radial_model, radial_best_c, radial_selection = select_regularization(
    X_train_radial, y_train, X_val_radial, y_val,
    candidates=[0.01, 0.1, 1.0, 10.0],
)
hybrid_model, hybrid_best_c, hybrid_selection = select_regularization(
    X_train_hybrid, y_train, X_val_hybrid, y_val,
    candidates=[0.001, 0.01, 0.1, 1.0],
)

print(f"Selected radial C: {radial_best_c}")
display(radial_selection)
print(f"Selected hybrid C: {hybrid_best_c}")
display(hybrid_selection)


Selected radial C: 0.01


,C,loss,accuracy,precision,recall,f1,roc_auc
0,0.01,0.647979,0.682051,0.648305,0.680000,0.663774,0.727713
1,0.10,0.650606,0.680000,0.641975,0.693333,0.666667,0.725066
2,1.00,0.652470,0.675897,0.635628,0.697778,0.665254,0.724438
3,10.00,0.652628,0.675897,0.635354,0.698889,0.665608,0.724385


Selected hybrid C: 0.01


,C,loss,accuracy,precision,recall,f1,roc_auc
0,0.010,0.554632,0.702051,0.724332,0.572222,0.639354,0.781760
1,0.100,0.557889,0.703590,0.718157,0.588889,0.647131,0.775878
2,0.001,0.560938,0.702564,0.731884,0.561111,0.635220,0.778708
3,1.000,0.563356,0.699487,0.708777,0.592222,0.645278,0.771443


In [9]:
frequency_results = {}
frequency_models = {
    "frequency_radial": (radial_model, X_test_radial),
    "frequency_hybrid": (hybrid_model, X_test_hybrid),
}

for model_name, (fitted_model, X_test) in frequency_models.items():
    probabilities = fitted_model.predict_proba(X_test)[:, 1]
    result = evaluate_probability_predictions(test_frame, probabilities, threshold=THRESHOLD)
    frequency_results[model_name] = result

    display(pd.Series(result.metrics.to_dict(), name=model_name))
    save_evaluation_outputs(
        result,
        output_dir=METRICS_ROOT / model_name,
        figures_dir=FIGURES_ROOT / model_name,
        model_label=model_name.replace("_", " ").title(),
    )
    joblib.dump(fitted_model, MODEL_ROOT / f"{model_name}.joblib")

radial_selection.to_csv(METRICS_ROOT / "frequency_radial" / "validation_selection.csv", index=False)
hybrid_selection.to_csv(METRICS_ROOT / "frequency_hybrid" / "validation_selection.csv", index=False)
print("Frequency outputs saved.")


loss         0.733056
accuracy     0.572195
precision    0.577750
recall       0.457000
f1           0.510329
roc_auc      0.596580
Name: frequency_radial, dtype: float64

loss         0.692354
accuracy     0.565854
precision    0.635468
recall       0.258000
f1           0.366999
roc_auc      0.678258
Name: frequency_hybrid, dtype: float64

Frequency outputs saved.


## 5. Preserve and extend the shared comparison

The existing per-fold CSV may already contain the colleague's ResNet18 and ViT rows. This cell performs an upsert: it replaces only rows for our three models and preserves every other model. It also rebuilds the cross-fold `logo_summary.csv`.


In [10]:
METRIC_NAMES = ["accuracy", "precision", "recall", "f1", "roc_auc"]
our_results = {
    "baseline_cnn": cnn_test_result.metrics.to_dict(),
    "frequency_radial": frequency_results["frequency_radial"].metrics.to_dict(),
    "frequency_hybrid": frequency_results["frequency_hybrid"].metrics.to_dict(),
}
seen_metric_paths = {
    "baseline_cnn": REPO_ROOT / "results" / "metrics" / "baseline_cnn" / "test_metrics.json",
    "frequency_radial": REPO_ROOT / "results" / "metrics" / "frequency" / "radial" / "test_metrics.json",
    "frequency_hybrid": REPO_ROOT / "results" / "metrics" / "frequency" / "hybrid" / "test_metrics.json",
}

new_rows = []
for model_name, unseen_metrics in our_results.items():
    new_rows.append({
        "model": model_name,
        "split": "unseen",
        **{metric: unseen_metrics[metric] for metric in METRIC_NAMES},
    })
    seen_path = seen_metric_paths[model_name]
    if seen_path.is_file():
        seen_metrics = json.loads(seen_path.read_text(encoding="utf-8"))
        new_rows.append({
            "model": model_name,
            "split": "seen",
            **{metric: seen_metrics[metric] for metric in METRIC_NAMES},
        })
    else:
        print(f"Missing standard metrics: {seen_path.relative_to(REPO_ROOT)}")

comparison_path = METRICS_ROOT / "seen_vs_unseen_comparison.csv"
new_comparison = pd.DataFrame(new_rows)
if comparison_path.is_file():
    existing_comparison = pd.read_csv(comparison_path)
    existing_comparison = existing_comparison[
        ~existing_comparison["model"].isin(our_results)
    ]
    comparison = pd.concat([existing_comparison, new_comparison], ignore_index=True)
else:
    comparison = new_comparison

comparison = comparison.sort_values(["model", "split"]).reset_index(drop=True)
comparison.to_csv(comparison_path, index=False)
display(comparison)

summary_frames = []
for path in sorted((REPO_ROOT / "results" / "metrics").glob("logo_*/seen_vs_unseen_comparison.csv")):
    per_run = pd.read_csv(path)
    per_run.insert(0, "unseen_generators", path.parent.name.removeprefix("logo_"))
    summary_frames.append(per_run)

logo_summary = pd.concat(summary_frames, ignore_index=True).sort_values(
    ["unseen_generators", "model", "split"]
)
summary_path = REPO_ROOT / "results" / "metrics" / "logo_summary.csv"
logo_summary.to_csv(summary_path, index=False)
print(f"Saved fold comparison: {comparison_path.relative_to(REPO_ROOT)}")
print(f"Updated summary: {summary_path.relative_to(REPO_ROOT)}")


,model,split,accuracy,precision,recall,f1,roc_auc
0,baseline_cnn,seen,0.796190,0.789033,0.808571,0.798683,0.887641
1,baseline_cnn,unseen,0.627805,0.715847,0.393000,0.507424,0.717647
2,frequency_hybrid,seen,0.708095,0.754366,0.617143,0.678889,0.794697
3,frequency_hybrid,unseen,0.565854,0.635468,0.258000,0.366999,0.678258
4,frequency_radial,seen,0.649048,0.618471,0.778095,0.689161,0.704762
5,frequency_radial,unseen,0.572195,0.577750,0.457000,0.510329,0.596580
6,resnet18,seen,0.819048,0.799107,0.852381,0.824885,0.906151
7,resnet18,unseen,0.765366,0.791246,0.705000,0.745637,0.848126
8,vit_small,seen,0.890000,0.880930,0.901905,0.891294,0.956613
9,vit_small,unseen,0.777561,0.868564,0.641000,0.737629,0.878077


Saved fold comparison: results\metrics\logo_wukong\seen_vs_unseen_comparison.csv
Updated summary: results\metrics\logo_summary.csv


## How to run the next fold

After one fold finishes and all outputs are saved:

1. Save the notebook.
2. Restart the kernel to release CNN and feature arrays.
3. Change only `UNSEEN_GENERATORS` in the setup cell.
4. Run all cells from the top again.

Standard seven-fold LOGO uses: `adm`, `biggan`, `glide`, `midjourney`, `sdv1_5`, `vqdm`, and `wukong`. The optional two-generator `sdv1_5+wukong` experiment can be repeated later, after all single-generator folds.
